In [137]:
#Import necessary libraries for data visualisation 

import pandas as pd
import numpy as np
import nbformat
from sqlalchemy import create_engine
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [138]:
#Connet to Heicoders Database 
ENDPOINT = 'heicoders-playground.c2ced10ceyki.ap-southeast-1.rds.amazonaws.com'
PORT = 3306
USERNAME = 'student300'
PASSWORD = 'heicoders_AI300'
DBNAME = 'ai300_capstone'
database_conn = create_engine(f'mysql+pymysql://{USERNAME}:{PASSWORD}@{ENDPOINT}/{DBNAME}')

query = """ 
SELECT
    a.account_id,
    a.contract_type,
    a.customer_id,
    a.tenure_months,
    a.num_referrals,
    a.has_internet_service,
    a.has_phone_service,
    a.has_multiple_lines,
    a.has_premium_tech_support,
    a.has_online_security,
    a.has_online_backup,
    a.has_device_protection,
    a.paperless_billing,
    a.payment_method,  -- All columns from the account table
    
    au.avg_long_distance_fee_monthly,
    au.total_long_distance_fee,
    au.avg_gb_download_monthly,
    au.stream_tv,
    au.stream_movie,
    au.stream_music,
    au.total_monthly_fee,
    au.total_charges_quarter,
    au.total_refunds,  -- All columns from the account_usage table excluding account_id
    
    c.gender,
    c.age,
    c.senior_citizen,
    c.married,
    c.num_dependents,
    c.zip_code,
    
    ct.area_id,
    ct.city,
    ct.latitutde,   
    ct.longitude,
    ct.population,
    
    cs.churn_label
    
FROM account as a
INNER JOIN account_usage AS au ON a.account_id = au.account_id
INNER JOIN customer AS c ON a.customer_id = c.customer_id
INNER JOIN city AS ct ON c.zip_code = ct.zip_code
INNER JOIN churn_status AS cs ON a.customer_id = cs.customer_id
WHERE churn_label = 'YES' or churn_label = 'NO';
"""

df = pd.read_sql(query, database_conn)
df


,account_id,contract_type,customer_id,tenure_months,num_referrals,has_internet_service,has_phone_service,has_multiple_lines,has_premium_tech_support,has_online_security,has_online_backup,has_device_protection,paperless_billing,payment_method,avg_long_distance_fee_monthly,total_long_distance_fee,avg_gb_download_monthly,stream_tv,stream_movie,stream_music,total_monthly_fee,total_charges_quarter,total_refunds,gender,age,senior_citizen,married,num_dependents,zip_code,area_id,city,latitutde,longitude,population,churn_label
0,AAJU-HMJLK,One Year,0334-ZFJSR,55,0,Yes,Yes,Yes,Yes,Yes,Yes,No,Yes,Credit Card,35.38,1945.90,13,No,No,No,66.05,3462.10,44.53,Female,41,No,Yes,0,92123,371,San Diego,32.808814,-117.134694,25232,No
1,ABBQ-EXMMW,Two Year,1820-DJFPH,72,4,No,Yes,Yes,No,No,No,No,Yes,Bank Withdrawal,48.29,3476.88,0,No,No,No,24.05,1709.15,0.00,Female,59,No,Yes,3,95555,1310,Orick,41.336354,-124.044354,494,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6989,ZMXR-RQYLT,One Year,4584-LBNMK,45,5,No,Yes,Yes,No,No,No,No,No,Credit Card,44.91,2020.95,0,No,No,No,24.70,1174.35,0.00,Male,79,Yes,Yes,0,96122,1607,Portola,39.786755,-120.445626,4236,No
6990,ZONO-OCVUP,Month-to-Month,2023-VQFDL,18,0,No,Yes,No,No,No,No,No,No,Bank Withdrawal,33.16,596.88,0,No,No,No,19.00,348.80,0.00,Male,42,No,No,0,92506,489,Riverside,33.930931,-117.361788,42425,No


In [139]:
#Checking for duplicated columns
duplicate_columns = df.T.duplicated()
#Display duplicated columns 
non_unique_columns = df.columns[duplicate_columns]
print("Non-unique columns:", non_unique_columns) #Duplicated columns are absent based on SQL Query, good. 

Non-unique columns: Index([], dtype='object')


In [140]:
#obtain summary of data
df.describe(include = 'all') #Presence of some categorical and nominal variables 

#Retrieve column with numbeers
desc = df.describe()
#Retrieve true numerical columns 
numerical_columns = ['tenure_months', 'num_referrals', 'avg_long_distance_fee_monthly',
       'total_long_distance_fee', 'avg_gb_download_monthly',
       'total_monthly_fee', 'total_charges_quarter', 'total_refunds', 'age',
       'num_dependents', 'population']

#Create new numerical_column_df 
numerical_column_df = df[numerical_columns]
numerical_column_df.describe() #View descriptive statistics for true numerical columns 
numerical_column_df.info() #Shows 11 numberical columns 

#Let's create some subplots for numerical_column_df
numerical_fig = make_subplots(rows = 4, cols = 3, subplot_titles = numerical_column_df.columns)
for i, col in enumerate(numerical_column_df.columns):
    row = (i // 3) + 1  # Determine the row
    col_number = (i % 3) + 1  # Determine the column
    # Add histogram trace
    numerical_fig.add_trace(
        go.Histogram(x=numerical_column_df[col], name=col),
        row=row, col=col_number
    )
numerical_fig.update_layout(height = 1000, width = 1200, title_text = "Subplots of Numeric Columns")
numerical_fig.show()

#Let's visualise the correlation matrix between these numerical columns as well 
correlation_matrix = df[numerical_columns].corr()
corr_num_fig = px.imshow(correlation_matrix, text_auto = True,
                         aspect = "auto",
                         title = "Correlation Heatmap of Numeric Columns ")
corr_num_fig.show()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6991 entries, 0 to 6990
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   tenure_months                  6991 non-null   int64  
 1   num_referrals                  6991 non-null   int64  
 2   avg_long_distance_fee_monthly  6991 non-null   float64
 3   total_long_distance_fee        6991 non-null   float64
 4   avg_gb_download_monthly        6991 non-null   int64  
 5   total_monthly_fee              6991 non-null   float64
 6   total_charges_quarter          6991 non-null   float64
 7   total_refunds                  6991 non-null   float64
 8   age                            6991 non-null   int64  
 9   num_dependents                 6991 non-null   int64  
 10  population                     6991 non-null   int64  
dtypes: float64(5), int64(6)
memory usage: 600.9 KB


In [141]:
#Alright, now let's look at the categorical and nominal variables as well, except customer_id and account_id 
cat_nom_subplots_df = df.drop(columns = numerical_column_df) #Remove numerical columns
cat_nom_subplots_df = cat_nom_subplots_df.drop(columns = ["account_id", "customer_id"]) #Remove account_id and customer_id 
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

#Let's make subplots for these figures, ignoring account_id and account_id 

cat_nom_subplots_fig = make_subplots(rows = 9, 
                                     cols = 4, 
                                     subplot_titles = cat_nom_subplots_df.columns,
                                     vertical_spacing = 0.06,
                                     horizontal_spacing = 0.06)


for i, col in enumerate(cat_nom_subplots_df.columns):
  row = (i//4) + 1
  col_number = (i%4) + 1
  #Add histogram trace
  cat_nom_subplots_fig.add_trace(go.Histogram(x=cat_nom_subplots_df[col], name = col), row=row, col=col_number)

cat_nom_subplots_fig.update_layout(height = 2000, 
                                   width = 1000, 
                                   title_text ="Subplots of Categorical and Nominal Features",
                                   showlegend = False)

cat_nom_subplots_fig



In [142]:
df.loc[df['churn_label'] == 1]

,account_id,contract_type,customer_id,tenure_months,num_referrals,has_internet_service,has_phone_service,has_multiple_lines,has_premium_tech_support,has_online_security,has_online_backup,has_device_protection,paperless_billing,payment_method,avg_long_distance_fee_monthly,total_long_distance_fee,avg_gb_download_monthly,stream_tv,stream_movie,stream_music,total_monthly_fee,total_charges_quarter,total_refunds,gender,age,senior_citizen,married,num_dependents,zip_code,area_id,city,latitutde,longitude,population,churn_label


In [143]:
#Let's do some feature engineering now :) 
#Select all dtypes = object and convert into string
df_column_names = df.select_dtypes(object).columns

#Change dtypes = object to dtypes = string
df[df_column_names] = df[df_column_names].astype("string") #Success 
pd.set_option('display.max_rows', 5)
pd.set_option('display.max_columns', 50)

In [144]:
#Change churn_label (target variable) to Yes 1's and 0's 
df['churn_label'] = df['churn_label'].map({'Yes':1, 'No':0})

#Changing has_internet_service to 0's and 1
df['has_internet_service'] = df['has_internet_service'].map({'Yes': 1, 'No': 0}) 

#Changing has_phone_service to 0's and 1's 
df['has_phone_service'] = df['has_phone_service'].map({'Yes': 1, 'No': 0}) 

#Convert has_multiple_lines to 0's and 1's 
df['has_multiple_lines'] = df['has_multiple_lines'].map({'Yes': 1, 'No':0})

#Convert has_premium_tech_support to 0's and 1's
df['has_premium_tech_support'] = df['has_premium_tech_support'].map({'Yes': 1, 'No': 0})

#Convert has_online_security to 0's and 1's 
df['has_online_security'] = df['has_online_security'].map({'Yes': 1, 'No': 0})

#Convert has_online_backup to 0's and 1's 
df['has_online_backup'] = df['has_online_backup'].map({'Yes': 1, 'No': 0})

#Convert has_device_protection into 0's and 1's 
df['has_device_protection'] = df['has_device_protection'].map({'Yes': 1, 'No': 0})

#Convert has_paperless_billing into 0's and 1's 
df['paperless_billing'] = df['paperless_billing'].map({'Yes': 1, 'No': 0}) 

#In hindsight, this probably wasn't necessary as CatBoost handles dichotomous variables 

In [145]:
df.loc[df['churn_label'] == 1]


,account_id,contract_type,customer_id,tenure_months,num_referrals,has_internet_service,has_phone_service,has_multiple_lines,has_premium_tech_support,has_online_security,has_online_backup,has_device_protection,paperless_billing,payment_method,avg_long_distance_fee_monthly,total_long_distance_fee,avg_gb_download_monthly,stream_tv,stream_movie,stream_music,total_monthly_fee,total_charges_quarter,total_refunds,gender,age,senior_citizen,married,num_dependents,zip_code,area_id,city,latitutde,longitude,population,churn_label
7,ACSF-MFCZF,Month-to-Month,2446-BEGGB,6,0,1,1,1,0,0,1,0,1,Bank Withdrawal,47.35,284.10,29,Yes,Yes,Yes,98.25,560.60,0.0,Female,79,Yes,No,0,90712,134,Lakewood,33.840524,-118.148403,30173,1
10,ADKI-PEMCS,Month-to-Month,2091-RFFBA,31,0,1,1,1,0,0,0,0,1,Bank Withdrawal,15.43,478.33,2,No,No,No,73.90,2217.15,0.0,Female,70,Yes,No,0,94569,973,Port Costa,38.035707,-122.196821,173,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6984,WKVR-GJRFI,Month-to-Month,4361-BKAXE,41,0,1,1,1,1,1,1,1,1,Bank Withdrawal,21.56,883.96,59,Yes,Yes,Yes,114.50,4527.45,0.0,Female,23,No,No,0,95006,1062,Boulder Creek,37.171727,-122.142961,10520,1
6986,YPVR-LUYBV,Month-to-Month,6328-ZPBGN,11,0,1,1,1,0,0,0,0,1,Bank Withdrawal,3.62,39.82,22,Yes,Yes,Yes,95.15,997.65,0.0,Female,67,Yes,No,0,95367,1207,Riverbank,37.734971,-120.954271,16525,1


Model Training


In [146]:
#Import necessary libraries for model training 
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn import metrics

In [147]:
#Feature selection - decided on easy to complete features 
features = ['age',
            'gender',
            'contract_type',
            'senior_citizen',
            'payment_method',
            'stream_tv', 
            'stream_movie',
            'stream_music']
X = df[features]
y = df['churn_label']

#Implement train_test_split 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.30, random_state =42)

In [148]:
# Implement CatBoost
model = CatBoostClassifier(learning_rate = 0.09444444444444444, l2_leaf_reg = 9, depth = 6, border_count = 100)

# Make sure categorical columns are correct and match the column names in X_train
model.fit(X_train, y_train, cat_features=['gender', 'contract_type', 'senior_citizen', 'payment_method', 
                                          'stream_tv', 'stream_movie', 'stream_music'], verbose=False)

# Predict probabilities for the test set
probability = model.predict_proba(X_test)
y_pred_probability = probability[:, 1]

# Calculate ROC and AUC
fpr, tpr, threshold = metrics.roc_curve(y_test, y_pred_probability)
AUC = metrics.auc(fpr, tpr)
print("AUC:", AUC)

AUC: 0.8359230177537409


In [149]:
#Create .pkl file for CatBoostModel
from pathlib import Path

Path("../model").mkdir(exist_ok=True)  # Create model/ directory if doesn't exist

import joblib

joblib.dump(model, '../model/catboost_model.pkl')

['../model/catboost_model.pkl']

In [150]:
#Let's try some hyper-parameter tuning
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score

param_dist = {
    'depth': np.arange(4, 10, 1),
    'learning_rate': np.linspace(0.01, 0.2, 10), 
    'l2_leaf_reg': np.arange(1, 10, 2),  
    'border_count': np.linspace(32, 100, 4, dtype=int)  
}
#Create model object
model = CatBoostClassifier(iterations = 100, verbose =0)

#Start RandomSearch to save time 
random_search = RandomizedSearchCV(estimator=model, param_distributions=param_dist, 
                                   n_iter=10, cv=3, scoring='roc_auc', verbose=1, random_state=42)

random_search.fit(X_train, y_train, cat_features=['gender', 'contract_type', 'senior_citizen', 'payment_method', 
                                          'stream_tv', 'stream_movie', 'stream_music'])

#Print best params and model
print("Best Parameters:", random_search.best_params_)
best_model = random_search.best_estimator_

y_pred_proba = best_model.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_pred_proba)
print("AUC:", auc)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Parameters: {'learning_rate': 0.09444444444444444, 'l2_leaf_reg': 9, 'depth': 6, 'border_count': 100}
AUC: 0.846494161713838


In [151]:
#Let's try a Logistic Regression with purely numerical features 
features = ['avg_long_distance_fee_monthly',
            'avg_gb_download_monthly',
            'total_monthly_fee',
            'total_charges_quarter',
            'total_refunds',
            'age',
            'population']

X = df[features]
y = df['churn_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.30, random_state =42)

logit_model = LogisticRegression(max_iter = 1000)
logit_model.fit(X_train, y_train)
probability = logit_model.predict_proba(X_test)
y_pred_probability = probability[:,1]
fpr,tpr,threshold = metrics.roc_curve(y_test, y_pred_probability)
AUC = metrics.auc(fpr,tpr)
print(AUC)


0.7863184875876698
